# 05 · Humanizer Agent

The voice stylist (RESPONDING). Code first turns the ranked result into **facts** with every
amount already formatted ("$38.50") and the reasons each listing scored well ("25% under
the market price", "near mint"). The LLM writes 2–3 short, casual sentences from them, the
way a friendly regular at a card shop would say it: the top pick with why, then the other two
together with their totals. The sample-data note is said once per conversation.
Then the **number check**: a regex pulls every dollar amount out of the reply, and each one
must appear in the facts; the top card's name must be there too.

On a failed check the node returns `number_check_passed = False` and the error; Pablo's
`route_after_responding` sends the graph back here once, and the retry prompt includes the
error. After that, or if the LLM fails, a fixed template is spoken. Policy Rule 4
(`enforce_number_check_before_speech`) runs before any reply is final.

The reply leads with what the persona cares about and mentions warnings, skipped risky
listings, estimated shipping, a budget taken from memory, and a widened search.

In [ ]:
#| default_exp agents.humanizer

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Humanizer Agent: turns the structured result into a short spoken reply, then checks every
dollar amount and card name against the data before anything is spoken (Section 3 & 6)."""

import json
import re
import time
from collections.abc import Callable
from typing import Any

from src.config import load_config
from src.llm import LLM, LLMError, make_llm, offline_handler
from src.pipeline import between
from src.schemas import CONDITION_WORDS, EvaluatedListing, ParsedQuery
from src.state import AgentState
from src.tools.policy import enforce_number_check_before_speech

## Number check

In [ ]:
#| export
MONEY = re.compile(r"\$\s?(\d[\d,]*(?:\.\d{1,2})?)")
PERSONA_WORDS = {
    "budget_parent": "Budget Parent",
    "serious_collector": "Serious Collector",
    "cautious_buyer": "Cautious Buyer",
}

In [ ]:
#| export
def fmt_usd(amount: float) -> str:
    return f"${amount:,.2f}"

In [ ]:
#| export
def cents(amount: str | float) -> int:
    if isinstance(amount, str):
        amount = float(amount.replace("$", "").replace(",", "").strip())
    return round(amount * 100)

In [ ]:
#| export
def number_check(text: str, allowed: list[float], required_names: list[str]) -> list[str]:
    """Problems with a reply; an empty list means it passed. Every dollar amount must be one
    of `allowed`, and every required card name must appear."""
    problems = []
    allowed_cents = {cents(a) for a in allowed}
    for m in MONEY.finditer(text):
        if cents(m.group(1)) not in allowed_cents:
            problems.append(f"amount ${m.group(1)} is not in the data")
    low = text.lower()
    for name in required_names:
        if name and name.lower() not in low:
            problems.append(f"card name '{name}' is missing")
    return problems

## Facts and the template

In [ ]:
#| export
# ---------------------------------------------------------------------------- facts
def card_facts(item: EvaluatedListing) -> dict[str, Any]:
    """One listing as pre-formatted strings, so the LLM never does arithmetic."""
    x, lc = item.listing, item.landed_cost
    if lc is None:
        raise ValueError(f"{x.listing_id} has no landed cost; RANKING must run first")
    if x.delivery_type == "in_person":
        shipping = "local pickup"
    else:
        shipping = fmt_usd(lc.shipping) + (" (estimated)" if lc.shipping_estimated else "")
    out = {
        "card": x.card_name,
        "set": x.set_name or x.set_code,
        "condition": CONDITION_WORDS.get(x.condition, x.condition),
        "price": fmt_usd(x.price),
        "shipping": shipping,
        "total": fmt_usd(lc.total),
        "city": x.location_city,
    }
    if item.reasons:
        out["why"] = item.reasons  # e.g. ["25% under the market price", "near mint"]
    if item.warning:
        out["warning"] = item.warning
    return out

In [ ]:
#| export
def build_facts(
    state: AgentState, config: dict[str, Any], sample_data: bool = False
) -> dict[str, Any]:
    """Everything the reply may say, as one JSON object."""
    q: ParsedQuery = state.get("parsed_query") or ParsedQuery()
    ranked = state.get("ranked_listings", [])
    rejected = state.get("rejected_listings", [])
    closest = state.get("closest_rejected")
    reject_at = config["thresholds"]["risk_reject"]

    notes = []
    if "budget" in q.filled_from_memory:
        notes.append("the budget was taken from the buyer's saved profile")
    if state.get("relaxed_filters"):
        notes.append("nothing safe matched exactly, so the search was widened")
    if sample_data and not state.get("sample_note_said"):  # once per conversation
        notes.append("these are sample listings and sample prices, not live data")
    risky = [x for x in rejected if x.risk_score >= reject_at]
    facts: dict[str, Any] = {
        "persona": PERSONA_WORDS[q.persona],
        "card_wanted": q.card_name,
        "budget": fmt_usd(q.budget) if q.budget is not None else None,
        "notes": notes,
        "skipped_risky": [
            {"card": x.listing.card_name, "reason": x.warning or "; ".join(x.risk_reasons[:2])}
            for x in risky
        ],
    }

    pc = state.get("price_check")
    if q.is_consultative and pc is not None:
        if not pc.get("found"):
            facts["status"] = "price_check_unknown"
            return facts
        facts["status"] = "price_check"
        facts["card"] = pc["card"]
        facts["set"] = pc.get("set")
        facts["condition"] = CONDITION_WORDS[pc["condition"]]
        facts["market_for_condition"] = fmt_usd(pc["market_for_condition"])
        if pc.get("condition_assumed"):
            notes.append("no condition was given, so lightly played was assumed")
        if pc.get("asked_price") is not None:
            facts["asked"] = fmt_usd(pc["asked_price"])
            facts["verdict"] = pc["verdict"]
        if ranked:
            facts["cheapest_safe_listing"] = card_facts(
                min(ranked, key=lambda x: x.landed_cost.total)
            )
        return facts

    if ranked:
        facts["status"] = "recommend"
        facts["top"] = card_facts(ranked[0])
        n_alt = config["response"]["max_alternatives"]
        facts["alternatives"] = [card_facts(x) for x in ranked[1 : 1 + n_alt]]
    elif closest is not None and closest.risk_score < reject_at:
        facts["status"] = "nothing_fits"
        facts["closest"] = card_facts(closest)
        if q.budget is not None and closest.landed_cost.total > q.budget:
            facts["why"] = f"it is {fmt_usd(closest.landed_cost.total - q.budget)} over your budget"
        else:
            facts["why"] = closest.rejection_reason or "it failed a safety rule"
    elif risky:
        facts["status"] = "nothing_fits"
        facts["why"] = "every matching listing looked risky"
    else:
        facts["status"] = "no_results"
    return facts

In [ ]:
#| export
def template_reply(facts: dict[str, Any]) -> str:
    """Fixed-wording fallback (Section 6), also used by offline mode."""
    reply = _template_body(facts)
    if any("saved profile" in n for n in facts.get("notes", [])) and facts.get("budget"):
        reply += f" I used your saved budget of {facts['budget']}."
    return reply

In [ ]:
#| export
def _template_body(facts: dict[str, Any]) -> str:
    status = facts.get("status")
    if status == "recommend":
        t = facts["top"]
        how = (
            " with local pickup"
            if t["shipping"] == "local pickup"
            else f", shipping {t['shipping']}"
        )
        lead = (
            f"Your best bet is the {t['card']} from {t['set']}, {t['condition']}, "
            f"for {t['total']}{how}"
        )
        why = [w for w in t.get("why", []) if w != t["condition"]][:2]  # condition is said already
        parts = [lead + (f": {_and(why)}." if why else ".")]
        if t.get("warning"):
            parts.append(f"Heads up: {t['warning'].rstrip('.')}.")
        alts = []
        for a in facts.get("alternatives", []):
            why = next((w for w in a.get("why", []) if w != a["condition"]), None)
            text = f"the {a['card']} ({a['condition']}) at {a['total']}" + (
                f", {why}" if why else ""
            )
            if a.get("warning"):
                w = a["warning"].rstrip(".")
                text += f", but {w[:1].lower()}{w[1:]}"
            alts.append(text)
        if alts:
            parts.append(f"Also worth a look: {'; or '.join(alts)}.")
        n = len(facts.get("skipped_risky", []))
        if n:
            parts.append(f"I skipped {n} listing{'s' if n > 1 else ''} that looked sketchy.")
        return " ".join(parts)
    if status == "nothing_fits":
        c = facts.get("closest")
        if c:
            return (
                f"No luck on a {facts['card_wanted']} that fits. The closest is the "
                f"{c['card']} in {c['condition']} condition at {c['total']}, but {facts['why']}."
            )
        return f"No luck on a safe {facts['card_wanted']}: {facts['why']}."
    if status == "price_check":
        lead = (
            f"{facts['asked']} for a {facts['condition']} {facts['card']} is {facts['verdict']}. "
            if "asked" in facts
            else ""
        )
        one = "one" if "asked" in facts else facts["card"]  # the card is named at least once
        return lead + (
            f"A {facts['condition']} {one} usually goes for about {facts['market_for_condition']}."
        )
    if status == "price_check_unknown":
        return f"I don't have a market price for {facts.get('card_wanted') or 'that card'} yet."
    return f"I couldn't find any {facts.get('card_wanted') or 'listings for that card'} right now."

In [ ]:
#| export
def _and(items: list[str]) -> str:
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]

In [ ]:
#| export
def allowed_amounts(facts: dict[str, Any]) -> list[float]:
    return [cents(m) / 100 for m in MONEY.findall(json.dumps(facts))]

In [ ]:
#| export
def required_names(facts: dict[str, Any]) -> list[str]:
    if facts.get("status") == "recommend":
        return [facts["top"]["card"]]
    if facts.get("status") == "price_check":
        return [facts["card"]]
    return []

## RESPONDING node

In [ ]:
#| export
HUMANIZER_SYSTEM = """You are a friendly regular at a local card shop, helping a buyer out loud.
Turn the facts into what you'd actually say: casual, warm and quick.
Rules:
- 2 or 3 short sentences, about 50 words at most. It is spoken, so no lists, markdown or emoji.
- Talk like a person: contractions, plain words, a quick reaction is fine ("Oh nice",
  "Good news"). No report phrasing: never "I recommend", "Please note", "Based on" or
  "Additionally".
- Use dollar amounts exactly as written in the facts. Never calculate, round or invent numbers.
- Mention only cards that appear in the facts, by their card name.
- Lead with the top card and why, in a few words from its "why" list. Then the alternatives
  together in one sentence, each with its total (and its warning, if it has one).
- A warning or skipped risky listings: one short, calm mention.
- Notes (estimated shipping, saved budget, widened search, sample data): a few words each, and
  only if they are in the notes.
- Lead with what the buyer's persona cares about: "Budget Parent" -> the total price;
  "Serious Collector" -> the condition; "Cautious Buyer" -> why the pick looks safe.
- If the status is "price_check", answer the question first: is the asked price fair compared
  with the market price for that condition? Mention a listing only after that."""

In [ ]:
#| export
def make_responding_node(
    config: dict[str, Any] | None = None,
    llm: LLM | None = None,
    check_numbers: bool = True,
    sample_data: bool = False,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the RESPONDING node. One LLM attempt per visit: if the number check fails, the
    graph comes back here once with the error (route_after_responding); after that, or if
    the LLM fails, the fixed template is used. Nothing is final until the check passes."""
    cfg = config or load_config()
    model = llm or make_llm(cfg)
    max_retries = int(cfg["limits"]["max_humanizer_retries"])

    def responding_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        facts = build_facts(state, cfg, sample_data)
        allowed, names = allowed_amounts(facts), required_names(facts)
        retries = int(state.get("humanizer_retry_count", 0))
        prompt = "Facts (JSON):\n<facts>\n" + json.dumps(facts, indent=1) + "\n</facts>\n"
        if state.get("number_check_error"):
            prompt += (
                f"Your previous reply had problems: {state['number_check_error']}. Fix them.\n"
            )
        prompt += "Spoken reply:"

        draft, problems, source = None, [], "llm"
        try:
            draft = re.sub(
                r"\s+", " ", model.text(tag="humanize", system=HUMANIZER_SYSTEM, prompt=prompt)
            )
            problems = number_check(draft, allowed, names) if check_numbers else []
        except LLMError as e:
            problems = [f"LLM failed: {e}"]

        timings = dict(state.get("stage_timings", {}))
        timings["RESPONDING"] = round(timings.get("RESPONDING", 0.0) + time.perf_counter() - t0, 4)
        if problems and draft is not None and retries < max_retries:
            return {
                "current_state": "RESPONDING",
                "draft_response": draft,
                "number_check_passed": False,
                "number_check_error": "; ".join(problems),
                "humanizer_retry_count": retries + 1,
                "stage_timings": timings,
            }
        final = draft
        if problems:
            final, source = template_reply(facts), "template"
            problems = number_check(final, allowed, names)
        enforce_number_check_before_speech(not problems)  # Policy Rule 4
        return {
            "current_state": "RESPONDING",
            "draft_response": draft,
            "final_response": final,
            "number_check_passed": True,
            "number_check_error": None,
            "response_source": source,
            "stage_timings": timings,
        }

    return responding_node

## Offline stand-in

In [ ]:
#| export
@offline_handler("humanize")
def offline_humanize(_system: str, prompt: str) -> str:
    """Offline mode speaks the template reply."""
    return template_reply(json.loads(between(prompt, "facts")))

# Try it

The number check catches a wrong price and a missing card name:

In [ ]:
print(number_check("The Charizard is $52.00 with local pickup.", [52.0], ["Charizard"]))
print(number_check("Grab it for $50, great deal!", [52.0], ["Charizard"]))

A full RESPONDING turn on a real ranked result (INTENT to RANKING run with the other agents):

In [ ]:
from src.agents.consumer import make_ranking_node
from src.agents.merchant import make_evaluating_node, make_searching_node
from src.config import load_config
from src.llm import make_llm
from src.pipeline import load_data
from src.schemas import ParsedQuery

cfg = load_config()
llm = make_llm(cfg)
data = load_data(cfg)
state = {
    "parsed_query": ParsedQuery(
        card_name="Charizard", set_code="Base Set", budget=400, persona="serious_collector"
    ),
    "current_state": "INTENT",
}
for node in (
    make_searching_node(data, cfg, live=False),
    make_evaluating_node(data, cfg, llm=llm),
    make_ranking_node(cfg),
):
    state |= node(state)
facts = build_facts(state, cfg, sample_data=True)
facts

In [ ]:
out = make_responding_node(cfg, llm=llm, sample_data=True)(state)
print(out["response_source"], "|", out["final_response"])
print("template would say:", template_reply(facts))

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()